# Single FCM Edge-List Quality Control

## Purpose

This notebook validates one aligned Functional Connectivity Matrix (FCM)
edge-list file from the Curvature-FCN-Aging dataset.

The validation checks the raw file format, data types, rows, nodes, ROI
pairs, missing values, and correlation range.

The final result will be a compact quality-control summary stating whether
the selected FCM file passes validation.



In [9]:
from pathlib import Path

import pandas as pd

from lemon_connectivity.alignment import align_participants_to_fcm
from lemon_connectivity.io import load_fcm_edge_list, summarize_fcm_edge_list


## Data source

The FCM files are stored in the Curvature-FCN-Aging dataset.
The cohort information file contains the participant IDs used for alignment.

In [2]:
# Define the folder containing the FCM files
fcm_data = Path(
    "../data/external/Curvature-FCN-Aging/DATA/FCM"
)

# Define the cohort metadata file
cohort_path = Path(
    "../data/external/Curvature-FCN-Aging/DATA/cohort_information.tsv"
)

## Load cohort metadata

We load the cohort information with `sub_id` as a string so that
participant identifiers are preserved exactly.

In [3]:
# Load the cohort metadata
cohort_metadata = pd.read_csv(
    cohort_path,
    sep="\t",
    dtype={"sub_id": "string"},
)

## Align participants with FCM files

The existing alignment function checks which cohort participants have
corresponding FCM files.

The alignment result will be used to select the FCM file reproducibly.

In [4]:
# Align cohort participants with the available FCM files
alignment_result = align_participants_to_fcm(
    cohort_metadata,
    fcm_data)

## Select one FCM file reproducibly

We use the alignment result from the previous step to identify FCM files
that are successfully aligned with cohort participants.

We then sort the valid aligned files alphabetically by their file path
and select the first one.

This makes the file selection deterministic and reproducible.

In [5]:
matched_records = alignment_result.alignment_table.loc[
    alignment_result.alignment_table["alignment_status"] == "matched"
].sort_values(
    ["canonical_id", "matrix_filename"],
    kind="stable")

if matched_records.empty:
    raise ValueError("No one-to-one participant/FCM matches are available")

selected_matrix_path = matched_records.iloc[0]["matrix_path"]
if pd.isna(selected_matrix_path):
    raise ValueError("The selected matched record has no matrix path")

target_file_path = Path(str(selected_matrix_path))
if not target_file_path.is_file():
    raise FileNotFoundError("The deterministically selected FCM file is unavailable")

In [6]:
edge_table = load_fcm_edge_list(target_file_path, n_rois=200)
edge_table

,roi_i,roi_j,correlation
0,9,109,0.911632
1,19,121,0.906548
2,9,108,0.902196
3,44,149,0.885611
4,108,112,0.882266
...,...,...,...
19895,44,81,-0.631550
19896,44,61,-0.632517
19897,61,154,-0.640387
19898,92,149,-0.654442


## Produce the QC summary table

The validation results are presented as a compact table showing the
expected value, observed value, and whether each quality-control check
passes.

In [7]:
qc_summary = summarize_fcm_edge_list(edge_table, n_rois=200)
qc_summary

,check,expected,observed,passed
0,Edge columns,"['roi_i', 'roi_j', 'correlation']","['roi_i', 'roi_j', 'correlation']",True
1,Edge rows,19900,19900,True
2,ROI data types,integer,"int64, int64",True
3,Correlation data type,numeric,float64,True
4,Unique nodes,200,200,True
5,ROI index set,0-199,0-199,True
6,Self-edges,0,0,True
7,Duplicate unordered pairs,0,0,True
8,Upper-triangle ordering,19900/19900 rows with roi_i < roi_j,19900/19900 rows,True
9,Complete unordered pairs,19900,19900,True


## Biological interpretation of one FCM edge

Each row in the FCM edge list represents a functional connection between
two brain regions.

For example, an edge connecting `roi_i = 12` and `roi_j = 57` with a
correlation of `0.42` means that the resting-state fMRI signals from
these two regions have a Pearson correlation of 0.42.

A positive correlation indicates that the two regional signals tend to
vary together, whereas a negative correlation indicates an inverse
relationship.

This represents functional/statistical connectivity and does not by
itself imply a direct anatomical connection or causal relationship.

## Final validation status

The FCM file passes validation only if all required quality-control
checks pass.

The final status is determined automatically from the validation report.

In [8]:
validation_passed = bool(qc_summary["passed"].all())
if not validation_passed:
    failed_checks = qc_summary.loc[~qc_summary["passed"], "check"].tolist()
    raise AssertionError(f"Single-file FCM QC failed: {failed_checks}")

final_status = pd.Series(
    {
        "status": "PASS",
        "checks_passed": int(qc_summary["passed"].sum()),
        "checks_total": len(qc_summary),
    },
    name="single_fcm_qc",
)
final_status.to_frame()

,single_fcm_qc
status,PASS
checks_passed,13
checks_total,13
